# 可视化分析：分布图

> 本章介绍金融数据分析中最常用的三类分布可视化工具：直方图、核密度函数图和箱线图/小提琴图。三者各有侧重，又相互补充——直方图是起点，核密度估计是其自然延伸，箱线图与小提琴图则以更紧凑的方式呈现分布的关键统计特征。

# 1. 直方图

## 1.1 基本原理

直方图（Histogram）是一种常用的数据可视化工具，用于展示数据的分布情况。通过将数据分成若干区间（bins），并统计每个区间中的数据点数目，以矩形的高度表示频数、频率或密度。直方图能够直观地反映数据的集中趋势、离散程度以及分布形态。

设有一组日收益率数据 $\{r_1, r_2, \ldots, r_n\}$，我们将其划分为 $K$ 个等宽的区间，每个区间的宽度为：

$$h = \frac{\max(r) - \min(r)}{K}$$

第 $k$ 个区间为 $[a_k, a_{k+1})$。绘图过程中，横轴表示收益率区间，纵轴有三种主要表示方式：

- **频数（Frequency）**：矩形的高度为 $n_k$，表示该区间内的数据点数量。
- **频率（Relative Frequency）**：矩形的高度为 $p_k = \frac{n_k}{n}$，表示该区间内数据点占总数据点的比例。有时也称为归一化频数或占比（Proportion）。
- **密度（Density）**：矩形的高度为 $f_k = \frac{p_k}{h} = \frac{n_k}{n \cdot h}$，表示单位区间内的频率。

In [ ]:
%reset -f

## 1.2 一个简单的例子

为了理解直方图的用途，我们先从柱状图开始，通过一个简单的例子理解「分布」的含义。某学习小组包括 10 名学生，年龄介于 16 岁到 26 岁之间。

In [ ]:
import numpy as np

ages = np.array([16] + [18] * 4 + [19] * 4 + [26])
print("Generated Ages:", ages)
print("Number of Students:", len(ages))

由于数据量很小，细心的读者可能已经算出了每个年龄的学生人数：16 岁 1 人，18 岁 4 人，19 岁 4 人，26 岁 1 人，甚至会列出如下表格：

| 年龄 | 16 | 18 | 19 | 26 |
|------|----|----|----|----|  
| 人数 |  1 |  4 |  4 |  1 |

通过这种方式，我们可以清楚地看到每个年龄段的学生人数分布情况。在制作上述表格的过程中，我们其实是对原数据进行了分组统计：将原数据分成四组，进而统计每组的人数。采用图形的方式可以更直观地展示上述信息：

![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250510003842.png)

## 1.3 频数、频率与密度

**频数**（Frequency）是指在某个区间内观察值的个数 ($n_k$)。

**频率** 是指该区间内观察值的个数 ($n_k$) 占总观察值个数 ($n$) 的比例。有时也会把「频率」称为「相对频数（Relative Frequency）」或「占比（proportion）」，计算公式为：

$$p_k = \frac{n_k}{n}$$

显然，频率的总和为 1，即 $\sum_{k=1}^{K} p_k = 1$。

**密度**（Density）是指单位区间内的频率，通常用于归一化处理。密度可以通过以下公式计算：

$$f_k = \frac{p_k}{h} = \frac{n_k}{n h}$$

其中，$f_k$ 是第 $k$ 个区间的密度，$h$ 是区间宽度。密度的总和不一定为 1，而是满足 $\sum_{k=1}^{K} f_k \cdot h = 1$。对于连续变量，当区间宽度趋近于 0 时，密度函数的极限就是概率密度函数（PDF）。

此外，绘制直方图时，若纵轴是频率，则取值范围为 $[0, 1]$；若纵轴是密度，则取值范围为 $[0, \infty)$（因为，当 $h$ 趋近于 0 时，$f_k$ 可以趋近于无穷大）。

从三者的定义也可以看出，无论纵轴为频数、频率还是密度，最终的直方图形状是一样的，只是纵轴的数值不同。

In [ ]:
import pandas as pd

# 计算唯一值及其计数
unique, counts = np.unique(ages, return_counts=True)

# 计算频率
frequencies = counts / counts.sum()

# 计算带宽 h
K = 4
h = (ages.max() - ages.min()) / K
print("Bandwidth (h):", h)

# 计算密度
density = frequencies / h

# 创建 DataFrame 包含计数、频率和密度
tabulated_ages = pd.DataFrame({
    "count": counts,
    "frequency": frequencies,
    "density": density
}, index=unique)

print(tabulated_ages)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(9, 3))

# 子图 1: 频数分布柱状图
axes[0].bar(unique, counts, color='skyblue', edgecolor='black')
axes[0].set_ylabel('Frequency')
axes[0].set_yticks(range(max(counts) + 1))

# 子图 2: 频率分布柱状图
axes[1].bar(unique, frequencies, color='lightgreen', edgecolor='black')
axes[1].set_ylabel('Relative Frequency')

# 子图 3: 密度分布柱状图
axes[2].bar(unique, density, color='salmon', edgecolor='black')
axes[2].set_ylabel('Density')

plt.tight_layout()
plt.show()

## 1.4 直方图的绘制

简单而言，绘制直方图的基本步骤为：

1. **选择区间数**：根据数据的范围和分布情况，选择合适的区间数 $K$。
2. **计算区间宽度**：根据数据的最大值和最小值，计算每个区间的宽度 $h$。
3. **统计频数**：统计每个区间内的数据点数量，得到频数 $n_k$。
4. **绘制直方图**：使用绘图工具将频数、频率或密度绘制成直方图。

>**Step 1：** 确定区间总数（$K$）

将数据划分为 $K$ 个区间。常见的选择区间总数的方法有：

- **经验法则**：通常取 $K = 10$ 或 $K=20$。

- **Sturges 法则**（Sturges' Rule）：
  $$K = \lceil \log_2 N + 1 \rceil$$
  其中，$\lceil z \rceil$ 表示对 $z$ 向上取整。

- **Freedman-Diaconis 法则**（Freedman-Diaconis Rule）：
  $$h = 2 \cdot \text{IQR} \cdot N^{-1/3}$$
  其中，$\text{IQR}$ 为四分位距。

- **Rice 法则**（Rice Rule，Stata 默认）：
  $$K = \min \left\{\sqrt{N},\ \frac{10 \ln(N)}{\ln(10)}\right\}$$
  该方法结合平方根法则和对数法则，适用于不同规模的数据集。当 $N<784$ 时，可直接采用 $\sqrt{N}$ 快速计算。

>**Step 2：** 确定区间宽度

$$h = \frac{\max(x) - \min(x)}{K}$$

>**Step 3：** 确定区间边界

设数据的最小值为 $x_{\min}$，最大值为 $x_{\max}$，则区间的边界可以表示为：

$$b_k = x_{\min} + (k-1) \cdot h \quad \text{for } k = 1, 2, \ldots, K+1$$

每个区间为 $[b_k, b_{k+1})$，最后一个区间为 $[b_K, b_{K+1}]$。

>**Step 4：** 统计每个区间的观察值个数

$$n_k = \sum_{j=1}^{N} \mathbf{1}(b_k \leq x_j < b_{k+1}) \quad \text{for } k = 1, 2, \ldots, K$$

其中 $\mathbf{1}(\cdot)$ 为指示函数，当条件为真时，取值为 $1$，否则取值为 $0$。

>**Step 5：** 绘制直方图

绘制直方图时，将每个区间的频数 $n_k$ 作为柱状图的高度，宽度为 $h$。当然，也可以根据需要用频率（$p_k$）或密度（$f_k$）来绘制直方图。

::: {.callout-note}
### 更为严谨的表述
> 参考资料：Wasserman (2006)《All of Nonparametric Statistics》，[PDF](https://www.stat.cmu.edu/~brian/valerie/617-2022/0%20-%20books/2006%20-%20Wasserman%20All%20Of%20Nonparametric%20Statistics.pdf)，第 6.2 节

直方图是最简单的非参数密度估计方法。假设密度函数 $f$ 的支撑集为某一区间，不失一般性，取该区间为 $[0,1]$。设 $m$ 为整数，并定义分组区间：

$$B_1=\left[0, \frac{1}{m}\right), \quad B_2=\left[\frac{1}{m}, \frac{2}{m}\right), \quad \ldots, \quad B_m=\left[\frac{m-1}{m}, 1\right]$$

定义组距 $h=1/m$，记 $Y_j$ 为落入 $B_j$ 中的观测个数，$\widehat{p}_j=Y_j/n$，$p_j=\int_{B_j} f(u)\,du$。

直方图估计量定义为：

$$\widehat{f}_n(x)=\sum_{j=1}^m \frac{\widehat{p}_j}{h}\, I\left(x \in B_j\right)$$

为理解该估计量的构造思想，注意到当 $x \in B_j$ 且 $h$ 较小时，有：

$$\mathbb{E}\left[\widehat{f}_n(x)\right]=\frac{\mathbb{E}\left(\widehat{p}_j\right)}{h}=\frac{p_j}{h}=\frac{\int_{B_j} f(u)\,du}{h} \approx \frac{f(x) h}{h}=f(x)$$

这是后续估计核密度函数的重要思想：通过**局部平均**来估计密度函数的值。
:::

## 1.5 Python 实操

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 生成年龄分布数据
np.random.seed(1234)
age = np.random.normal(loc=21, scale=7, size=100).astype(int)
age = age[(age >= 16) & (age <= 35)]

# 绘制直方图
plt.figure(figsize=(3, 3))
plt.hist(age, edgecolor='black', alpha=0.7)
plt.show()

本例中，我们只在 `plt.hist()` 函数中指定了变量名 `age`，而没有指定 `bins` 和 `rwidth` 参数。此时，函数会自动选择合适的区间数量和宽度。根据数据的分布情况，函数会将数据划分为 10 个区间，并计算每个区间内的数据点数量。

我们也可以自行指定 `bins` 和 `rwidth` 参数。如下命令的效果与上面相同：

```python
K = 10                           # Number of bins
h = (age.max() - age.min()) / K  # Bandwidth

# 指定 bins 数量
plt.hist(age, bins=K,   edgecolor='black', alpha=0.7)

# 指定 rwidth
plt.hist(age, rwidth=h, edgecolor='black', alpha=0.7)
```

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(6, 2))

K_values = [4, 10, 20]

for i, K in enumerate(K_values):
    axes[i].hist(age, bins=K, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'K={K}')
    axes[i].set_yticks(range(0, 32, 5))

plt.tight_layout()
plt.show()

从上图中可以看出：

- 不同的 bins 会导致直方图的分组方式不同，从而影响数据分布的可视化效果。较少的 bins 会导致信息的过度简化，而较多的 bins 则可能使图形过于复杂，难以观察整体趋势。

- 为了增加对比，我们设定了 `axes[i].set_yticks(range(0, 32, 5))`。可以看出，当我们划分的组数较多时（`bins` 值较大），落入每个区间的数据点数量自然会相对变少，导致直方图的高度不均匀，且可能出现一些区间的高度为 0 的情况。因此，过大的 `bins` 值虽然能够提供更精细的分布信息，但也可能导致我们「只见树木，不见森林」。

多数情况下，我们都无需手动指定 `bins` 和 `rwidth` 参数，直接使用 `plt.hist()` 函数自动选择的最优值即可。

### `plt.hist()` 函数详解

`plt.hist()` 函数是 `matplotlib` 库中用于绘制直方图的函数。其基本语法如下：

```python
plt.hist(x, bins=None, range=None, density=False, 
         weights=None, cumulative=False, bottom=None, 
         histtype='bar', align='mid', orientation='vertical', 
         rwidth=None, 
         color=None, edgecolor=None, alpha=None, 
         label=None, stacked=False, **kwargs)
```

其中，常用参数如下：

- `x`：表示要绘制直方图的数据，可以是列表、数组或 `pandas` 的 `Series` 对象。
- `bins`：表示区间的数量或边界，可以是整数或列表。若为整数，则表示将数据划分为 `bins` 个等宽区间；若为列表，则表示指定每个区间的边界。如 `bins=20`，或 `bins=[-0.1, -0.05, 0, 0.05, 0.1]`。
- `density`：布尔值，表示是否将直方图标准化为概率密度（面积为 1）。默认为 `False`。
- `weights`：表示每个数据点的权重，可以是与 `x` 等长的数组。
- `cumulative`：布尔值，表示是否绘制累积直方图。默认为 `False`。
- `bottom`：表示每个柱子的底部位置，可以是与 `x` 等长的数组。
- `histtype`：表示直方图的类型，可以是 `'bar'`、`'step'` 或 `'stepfilled'`。
- `align`：表示柱子的对齐方式，可以是 `'left'`、`'mid'` 或 `'right'`。
- `orientation`：表示柱子的方向，可以是 `'vertical'` 或 `'horizontal'`。
- `rwidth`：表示柱子的宽度，可以是一个浮点数，表示相对于区间宽度的比例。
- `color`：表示柱子的颜色，可以是字符串或 RGB 值。
- `edgecolor`：表示柱子的边框颜色。
- `alpha`：表示柱子的透明度，可以是一个浮点数，范围在 0 到 1 之间。
- `label`：表示图例标签。
- `stacked`：布尔值，表示是否堆叠直方图。默认为 `False`。
- `kwargs`：其他参数，可以传递给 `matplotlib` 的绘图函数，例如 `figsize=(10, 6)`。

## 1.6 注意事项

在比较直方图时，为了便于观察差异，建议将直方图垂直排列，以便更直观地观察横向变化。例如，尝试比较图中顶部的两个直方图。若将两个直方图水平排列，通常会因横向偏移而难以识别它们之间的差异。

![直方图对比示例](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250508000119.png)

> 图：当排列直方图以便于比较时，建议垂直对齐以观察横向变化。

## 1.7 扩展阅读

### 直方图实例

This Python code creates a **histogram** using the [Matplotlib](https://python-graph-gallery.com/matplotlib/) library to visualize data about *salaries in France*. It was originally produced by the [INSEE](https://www.insee.fr/fr/accueil).

<img style="width: 500px" src="https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250511114743.png">

> Source: [Histogram with custom style and annotations](https://python-graph-gallery.com/web-histogram-with-annotations/)

### Python
- [matplotlib - Histogram bins, density, and weight](https://matplotlib.org/stable/gallery/statistics/histogram_normalization.html)
- [Scatter plot with histograms](https://matplotlib.org/stable/gallery/lines_bars_and_markers/scatter_hist.html#sphx-glr-gallery-lines-bars-and-markers-scatter-hist-py)
- [seaborn - distribution](https://seaborn.pydata.org/tutorial/distributions.html)

### Stata
- 万莉, 2020, [Stata绘图全解：绘图语法-条形图-箱型图-散点图-矩阵图-直方图-点图-饼图](https://www.lianxh.cn/details/34.html), 连享会 No.34.
- 万莉, 2020, [Stata：读懂直方图](https://www.lianxh.cn/details/479.html), 连享会 No.479.
- 刘欣妍, 史柯, 2022, [Stata：描述性统计分析新命令-dstat](https://www.lianxh.cn/details/926.html), 连享会 No.926.
- 孙晓艺, 2024, [Stata绘图大礼包：27个常用的可视化范例及代码](https://www.lianxh.cn/details/1372.html), 连享会 No.1372.
- 汪京, 2024, [multihistogram-多变量直方图](https://www.lianxh.cn/details/1457.html), 连享会 No.1457.
- 袁子晴, 2021, [史上最牛Stata绘图模板-schemepack：酷似R中的ggplot2](https://www.lianxh.cn/details/819.html), 连享会 No.819.
- 谢嘉伟, 2024, [Stata 绘图：binscatterhist-分仓散点图+直方图](https://www.lianxh.cn/details/1506.html), 连享会 No.1506.
- 郑宇, 2024, [Stata绘图：加权直方图](https://www.lianxh.cn/details/1425.html), 连享会 No.1425.

---

# 2. 核密度函数图

## 2.1 从直方图到核密度估计

直方图有一个明显的缺陷：估计结果依赖于区间的起点（$t_0$）选择。相同的数据、相同的带宽，仅因起点不同，图形可能大相径庭。

**改进方向一：移动直方图**

一种自然的改进是采用「移动窗口」：对每个估计点 $x$，以 $x$ 为中心，统计落在 $[x - h/2,\ x + h/2]$ 内的数据点，计算密度估计值：

$$\hat{f}(x) = \frac{1}{nh} \sum_{i=1}^{n} \mathbf{1}\left\{|X_i - x| \leq \frac{h}{2}\right\}$$

这已是核密度估计的雏形——窗口内所有点享有相同权重，不在窗口内的点权重为零。但权重分配仍然过于「硬」：边界处权重从 1 骤降为 0。

**改进方向二：引入核函数，用平滑权重替代硬边界**

更优雅的做法是用一个光滑函数来替代硬边界，根据数据点 $X_i$ 与估计点 $x$ 之间的**距离**来分配权重——距离越近，权重越大；距离越远，权重平滑衰减，而不是突然截断为零。这个分配权重的函数就是**核函数**（Kernel Function），记为 $K(\cdot)$。

## 2.2 核密度估计

核密度估计（Kernel Density Estimation, KDE）是一种用于估计未知概率密度函数的非参数方法，适用于连续型数据且不依赖于事先指定的分布形式。

设样本为 $x_1, x_2, \dots, x_n$，其密度函数在任意点 $x$ 上的估计形式为：

$$\hat{f}_h(x) = \frac{1}{n h} \sum_{i=1}^{n} K\!\left( \frac{x - x_i}{h} \right)$$

其中：

- $K(\cdot)$ 是**核函数**（kernel function），通常是一个对称的概率密度函数；
- $h > 0$ 是**带宽参数**（bandwidth），控制核函数的缩放程度和平滑水平；
- $\hat{f}_h(x)$ 是点 $x$ 处的密度估计值。

核函数的作用可以理解为：在估计点 $x$ 处，根据样本点 $x_i$ 与 $x$ 之间的距离，赋予不同的权重。距离 $x$ 越近的样本点，其权重越大；距离越远，权重越小。通过对所有样本点的加权平均，得到该点的密度估计。将所有位置的估计值拼接起来，即可得到整体的密度函数曲线。

为了更清楚地理解核函数的加权机制，我们可以对距离进行标准化处理，设：

$$u_i = \frac{X_i - c}{h}$$

则以下两式等价：

$$|u_i| \leq 1 \Longleftrightarrow |X_i - c| \leq h$$

记 $D_i = |X_i - c|$，表示第 $i$ 个观察值与估计点 $c$ 的距离。核函数的任务就是为每个 $D_i$ 分配权重。

如下图所示，三种典型核函数的权重分配机制具有显著差异：

<img style="width: 600px" src="https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Fig-NP-kernel-fn01.png">

- **Uniform 核**：在 $|u| \leq 1$ 范围内赋予所有观察值相同的权重，超出范围的样本点权重为 0（相当于弃之不用）。对应的密度估计不具有平滑性，常用于教学演示。
- **Triangle 核**：采用线性下降的加权方式，距离估计点越近权重越大，边界处权重为 0，估计结果具有一定的连续性。
- **Epanechnikov 核**：采用抛物线型权重函数，在 $u=0$ 处取得最大值，具有最小均方误差（MSE）性质，估计曲线光滑、效率较高。
- **Gaussian 核**：采用正态分布函数，所有样本点均有非零权重，平滑程度高，适用于大多数实际应用场景。

## 2.3 核函数的性质

**常见核函数及其表达式：**

- **Uniform 核函数**：$K(u) = \frac{1}{2} \cdot \mathbf{1}\{|u| \leq 1\}$（也称为 Rectangular 核函数）
- **Triangle 核函数**：$K(u) = (1 - |u|) \cdot \mathbf{1}\{|u| \leq 1\}$
- **Epanechnikov 核函数**：$K(u) = \frac{3}{4}(1 - u^2) \cdot \mathbf{1}\{|u| \leq 1\}$
- **Quartic 核函数**：$K(u) = \frac{15}{16}(1 - u^2)^2 \cdot \mathbf{1}\{|u| \leq 1\}$
- **Triweight 核函数**：$K(u) = \frac{35}{32}(1 - u^2)^3 \cdot \mathbf{1}\{|u| \leq 1\}$
- **Gaussian 核函数**：$K(u) = \frac{1}{\sqrt{2\pi}} \exp\!\left(-\frac{u^2}{2}\right)$
- **Cosinus 核函数**：$K(u) = \frac{\pi}{4} \cos\!\left(\frac{\pi}{2} u\right) \cdot \mathbf{1}\{|u| \leq 1\}$

<img style="width: 600px" src="https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/Fig-NP-kernel-fn02.png">

核函数通常需要满足以下数学性质：

1. 非负性：$K(u) \geq 0$
2. 单位积分：$\int_{-\infty}^{\infty} K(u)\,du = 1$
3. 对称性：$K(u) = K(-u)$
4. 有限的二阶矩：$\int u^2 K(u)\,du < \infty$

实际使用中，部分文献或软件将 $\mathbf{1}\{|u| \leq 1\}$ 写为 $\mathbf{1}\{|u| < 1\}$。对于连续变量，两者几乎没有区别；但若数据是离散型的（如整数型变量），则可能影响边界值是否被纳入计算。

核密度估计的构造可以理解为：以每一个样本点为中心放置一个缩放后的核函数，然后在每一个估计位置 $x$ 上，取所有样本点的核值加权平均。因此，它是一种基于样本加权「局部贡献」的整体平滑过程。

总结而言：

- 核函数定义了如何根据样本点与估计点之间的距离分配权重；
- 带宽参数决定了每个样本点的影响范围；
- 合理选择核函数和带宽参数是核密度估计中最关键的步骤；
- 核密度估计为我们提供了一种平滑、灵活且无需模型假设的分布估计方法，广泛应用于经济学、金融学、机器学习等领域的探索性数据分析任务中。

在实际应用中，**核函数的选择对估计结果的影响相对较小，而带宽的设置对估计曲线的光滑程度影响较大**。

## 2.4 单变量核密度函数图

::: {.callout-tip}
### 提示词

目的：生成模拟数据，绘制核密度函数图
- 语言：Python
- N = 1000, x ~ N(10, 3), lnx = ln(x)
- 绘制 x 和 lnx 的核密度函数图
- 布局：1 行 2 列
:::

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 生成模拟数据
N = 1000
np.random.seed(142)
x = np.random.normal(loc=10, scale=3, size=N)
lnx = np.log(x)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
sns.kdeplot(x, label='x ~ N(10, 3)')
plt.title('KDE of x')
plt.xlabel('x'); plt.ylabel('Density')
plt.legend()

plt.subplot(1, 2, 2)
sns.kdeplot(lnx, label='ln(x)', color='orange')
plt.title('KDE of ln(x)')
plt.xlabel('ln(x)'); plt.ylabel('Density')
plt.legend()

plt.tight_layout()
plt.show()

## 2.5 多变量核密度函数图

- 不同时期的收入分布 - 时序
- 不同种族的收入分布 - 截面
- 联合分布

In [ ]:
!pip install pyreadstat requests

In [ ]:
import requests
import pyreadstat

url = "https://www.stata-press.com/data/r17/nlsw88.dta"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)
with open("data/nlsw88.dta", "wb") as f:
    f.write(r.content)

df, meta = pyreadstat.read_dta("data/nlsw88.dta")
print(df.head())

::: {.callout-tip}
### 提示词

- 绘制 White (race==1) 和 Black (race==2) 的 wage 变量的核密度函数图
- 尺寸：5 x 3
- 布局：1 行 1 列，将两个核密度函数图叠加
- 颜色：White = blue, Black = red (lpattern: dash)
- 透明度：0.8
- 标题字号: 12, 图例字号: 10, 横轴和纵轴字号: 10
:::

In [ ]:
plt.figure(figsize=(5, 3))
sns.kdeplot(df.loc[df['race'] == 1, 'wage'],
            label='White', color='blue', alpha=0.8)
sns.kdeplot(df.loc[df['race'] == 2, 'wage'],
            label='Black', color='red', alpha=0.8, linestyle='--')
plt.title('KDE of Wage: White vs Black', fontsize=12)
plt.xlabel('Wage', fontsize=10); plt.ylabel('Density', fontsize=10)
plt.legend(fontsize=10)
plt.xticks(fontsize=10); plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

**练习：** 绘制 White 和 Black 两个种族的妇女的 ln(wage) 的核密度函数图。

### 多变量核密度函数图：山脊图

::: {.callout-tip}
### 提示词 - v2
```
请帮我写一段 Python 代码，使用 `akshare` 下载多只 A 股股票在 `2024` 年的日度数据，
并绘制这些股票日对数收益率的山脊图。

使用的库包括：akshare、pandas、matplotlib.pyplot、numpy、joypy。

股票清单如下：
  中国移动：sh600941    贵州茅台：sh600519
  万科 A：  sz000002    比亚迪：  sz002594
  宁德时代：sz300750    南方航空：sh600029    格力电器：sz000651

设定年份为 2024，并自动生成起止日期：20240101 和 20241231。
若某只股票没有数据或下载出错，请打印提示并跳过，不要让程序中断。
代码要简洁、清楚，加上必要的中文注释，不要封装成函数，直接给出完整代码。
```
:::

In [ ]:
!pip install akshare joypy

In [ ]:
import akshare as ak
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import joypy

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size'] = 12

stock_dict = {
    '中国移动': 'sh600941', '贵州茅台': 'sh600519',
    '万科A':   'sz000002', '比亚迪':   'sz002594',
    '宁德时代': 'sz300750', '南方航空': 'sh600029', '格力电器': 'sz000651'
}

year = 2024
start_date = f'{year}0101'
end_date   = f'{year}1231'

returns_list = []

for name, code in stock_dict.items():
    try:
        df_s = ak.stock_zh_a_daily(symbol=code, start_date=start_date, end_date=end_date)
        if df_s.empty:
            print(f"{name}（{code}）在 {year} 年无数据，跳过。")
            continue
        df_s = df_s[['date', 'close']].copy()
        df_s['date'] = pd.to_datetime(df_s['date'])
        df_s.sort_values('date', inplace=True)
        df_s['close'] = df_s['close'].astype(float)
        df_s['log_ret'] = np.log(df_s['close']).diff()
        df_s['stock'] = name
        returns_list.append(df_s[['date', 'log_ret', 'stock']])
    except Exception as e:
        print(f"下载 {name}（{code}）时出错：{e}")

if len(returns_list) > 0:
    returns_df = pd.concat(returns_list, axis=0)
    returns_df = returns_df.dropna(subset=['log_ret'])
    data_long = returns_df[['stock', 'log_ret']]

    joypy.joyplot(data_long, by='stock', column='log_ret',
                  figsize=(12, 8), bins=50, range_style='own',
                  legend=False, overlap=1, linewidth=1, fade=True)
    plt.title(f"{year} 年多只股票日收益率分布山脊图", fontsize=18)
    plt.xlabel("日对数收益率", fontsize=14)
    plt.ylabel("股票名称", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("未能获得任何股票数据，无法绘图。")

In [ ]:
# 平面图 + 叠加密度图
plt.figure(figsize=(10, 6))
for stock_name in returns_df['stock'].unique():
    sub = returns_df[returns_df['stock'] == stock_name]['log_ret']
    sub.plot(kind='kde', label=stock_name)
plt.title(f"{year} 年多只股票日对数收益率分布图")
plt.xlabel("日对数收益率"); plt.ylabel("密度")
plt.legend(loc='upper right', fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2.6 进阶用法

### 散点图 + 核密度函数图（jointplot）

**penguins** 数据集是一个经典的数据集，包含了三种企鹅物种（Adelie、Chinstrap 和 Gentoo）的生物测量数据。本例中，我们使用 seaborn 的 `jointplot` 函数绘制散点图和核密度函数图。散点图展示了每个物种的鳍长和喙长的分布情况，而核密度函数图则呈现了横轴变量（`flipper_length_mm`）和纵轴变量（`bill_length_mm`）的分布情况。

散点图反映了企鹅的喙长（$y$）与鳍长（$x$）之间的关系。可以看出，虽然三种企鹅的 $y \sim x$ 之间都是正相关的，但 Chinstrap 类和 Gentoo 类企鹅的喙长和鳍长之间的正相关关系更强一些，而 Adelie 类企鹅的正相关关系相对较弱。

密度函数图则揭示了单变量的分布特征：

- Adelie：体型较小，喙长和鳍长都较小（「短嘴短翅」）。
- Chinstrap：体型中等，喙长与 Gentoo 差不多，鳍长与 Adelie 相近（「长嘴短翅」）。
- Gentoo：体型较大，喙长和鳍长都较大（「长嘴长翅」）。若进一步结合散点图，则可以推断 Gentoo 类企鹅群体身材较为均匀，而 Adelie 类企鹅个体之间的身材差异较大。

In [ ]:
import seaborn as sns

penguins = sns.load_dataset("penguins")

sns.jointplot(
    data=penguins,
    x="flipper_length_mm",
    y="bill_length_mm",
    hue="species",
    height=6
)

> 它们到底长啥样？  
> 你可以在 [这里](https://www.animalspot.net/penguin) 找到它们的详细介绍。有位热心的[网友](https://www.deviantart.com/bluegio/art/Adelie-Chinstrap-and-Gentoo-867439280)特意绘制了它们的合影（从左到右依次为 Adelie、Chinstrap 和 Gentoo），如下图所示：

<img src="figs/graph-Adelie-Chinstrap-and-Gentoo-867439280.jpg" width="500pt">

### 多个变量的情形：pairplot

在上面的例子中，我们只考虑了两个变量之间的关系。实际上，数据集中可能有多个变量，此时可以使用 seaborn 中的 `pairplot` 函数来绘制多个变量之间的关系图。

下面的例子中，我们同时呈现了三个变量（`flipper_length_mm`、`bill_length_mm` 和 `body_mass_g`）之间的两两配对散点图，以及单个变量的核密度函数图。与此同时，我们还使用了不同的颜色来区分不同的物种。

In [ ]:
# Source: https://seaborn.pydata.org/tutorial/introduction.html
vlist = ["flipper_length_mm", "bill_length_mm", "body_mass_g"]
sns.pairplot(data=penguins, vars=vlist, hue="species", corner=True)

### 3D 核密度函数图

![](https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/20250511164056.png)

> Source: [matplotlib - Fill under 3D line graphs](https://matplotlib.org/stable/gallery/mplot3d/fillunder3d.html#sphx-glr-gallery-mplot3d-fillunder3d-py)

### 密度函数图 + 条形码（rugplot）

在密度曲线下方叠加条形码（rug），可以同时看到整体分布和每个实际观测值的位置。

参考：[seaborn.rugplot](https://seaborn.pydata.org/generated/seaborn.rugplot.html)

In [ ]:
import seaborn as sns; sns.set_theme()
tips = sns.load_dataset("tips")
sns.kdeplot(data=tips, x="total_bill")
sns.rugplot(data=tips, x="total_bill")

---

# 3. 箱线图与小提琴图

数据分析过程中，均值和标准差只能粗略地描述数据的集中趋势和离散程度，但不能反映数据的分布形态、偏态和峰态等特征。本节介绍两种常用的可视化工具：箱线图（Box Plot）和小提琴图（Violin Plot）。它们能够直观地呈现中位数、25% 分位数、75% 分位数等统计量，更为全面地描述数据的分布特征和可能存在的离群值。

In [ ]:
%reset -f

import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## 3.1 箱线图（Boxplot）

<img style="width: 550px" src="https://fig-lianxh.oss-cn-shenzhen.aliyuncs.com/box_plots_01.png">

对于变量 $x$，我们将其中位数记为 $p50$ 或 $Q_2$，第一四分位数（$p25$）和第三四分位数（$p75$）分别记为 $Q_1$ 和 $Q_3$。同时，其最大值和最小值分别记为 $x_{\max}$ 和 $x_{\min}$。

箱线图由箱体、胡须和异常值三部分组成。

**箱体：**
- 箱体的上下边缘分别是数据的第一四分位数（$Q_1$）和第三四分位数（$Q_3$）。
- 中间的横线表示数据的中位数（Median）。
- 箱体的高度表示数据的四分位距（IQR），即 $\text{IQR} = Q_3 - Q_1$。

**胡须：**
- 箱体的上胡须延伸至 $B^H = Q_3 + 1.5 \times \text{IQR}$ 的位置（若 $x_{\max} < B^H$，则上胡须延伸至 $x_{\max}$）。
- 箱体的下胡须延伸至 $B_L = Q_1 - 1.5 \times \text{IQR}$ 的位置（若 $x_{\min} > B_L$，则下胡须延伸至 $x_{\min}$）。

**异常值：**
- 异常值是指超出胡须范围的观测值，通常用圆圈表示。

此处，$1.5 \times \text{IQR}$ 是一个常用的经验值，用于判断数据的异常值。参数 1.5 取决于我们对异常值的定义，通常取值范围在 1.5 到 3 之间。取值越大，表示我们对异常值的定义越宽松。

下图展示了**箱线图结构（上图）与正态分布概率密度函数（下图）之间的对应关系**。

- 红色区域覆盖中间的 50%，对应箱体内部；
- 两侧蓝色区域各占 24.65%，与胡须区间一致；
- 超出 $\pm 2.698\sigma$ 的区域仅占 0.35%，对应箱线图之外的极端值。

<img style="width: 550px" src="./figs/box_plots_02.png">

## 3.2 直观感受

下面，我们模拟生成一个服从 $N(0,1)$ 分布的随机数，$N=5000$，然后分别绘制其直方图（核密度函数图）、箱线图和小提琴图。

```
📊 数据分布摘要
├─ 核心趋势
│  ├─ 均值：  0.02
│  └─ 中位数：0.03
├─ 四分位距
│  ├─ Q1：-0.65
|  ├─ Q3： 0.69
│  └─ IQR：1.34 (Q3-Q1)
├─ 理论边界
│  ├─ 下限：Q1-1.5IQR = -2.66
│  └─ 上限：Q3+1.5IQR =  2.70
└─ 实际极值
   ├─ 最小值：-3.80
   └─ 最大值： 3.57
```

![](figs/graph_boxplot_violin_simu_Normal.png)

## 3.3 小提琴图（Violin Plot）

小提琴图是箱线图的扩展，除了展示数据的分布特征外，还能显示数据的密度分布。它通过在箱线图的基础上添加一个核密度估计（KDE）曲线来实现。小提琴图可以更好地揭示数据的分布形态，尤其是在数据量较大时。

小提琴图的核心部分与箱线图类似，但还包含了以下要素：

- **核密度估计（KDE）**：小提琴图的两侧展示了数据的密度分布，通常使用高斯核密度估计来平滑数据分布。
- **小提琴形状**：小提琴图的形状表示数据的分布特征，宽度越大表示数据在该位置的密度越高。

## 3.4 箱线图与小提琴图的对比

箱线图和小提琴图都是用于展示数据分布的可视化工具，但它们在信息传达和视觉效果上有所不同：

- **信息传达**：箱线图主要关注数据的集中趋势和离散程度，而小提琴图则同时展示了数据的分布形态和密度信息。
- **视觉效果**：箱线图通常较为简洁，适合快速识别数据的基本特征；小提琴图则提供了更丰富的信息，但可能在视觉上显得复杂。
- **数据量**：在数据量较小的情况下，箱线图可能更易于理解；而在数据量较大的情况下，小提琴图能够更好地揭示数据的分布特征。

## 3.5 模拟分析：四种典型分布

模拟四个序列：N = 200，Python，seed = 42，分布如下：
1. 标准正态分布
2. 左偏分布，有少量离群值 (10%)
3. 右偏分布，有少量离群值 (10%)
4. 对称分布，有大量离群值 (30%)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import skew, kurtosis
from tabulate import tabulate
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
N = 200
data = {
    "x1": np.random.normal(0, 1, N),
    "x2": np.concatenate([np.random.exponential(1, N-10), np.random.normal(-3, 0.5, 10)]),
    "x3": np.concatenate([np.random.exponential(1, N-10)*-1, np.random.normal(3, 0.5, 10)]),
    "x4": np.concatenate([np.random.normal(0, 1, N-30), np.random.normal(0, 5, 30)])
}

stats = {}
for name, values in data.items():
    stats[name] = {
        "Mean": np.mean(values), "SD": np.std(values),
        "Min":  np.min(values),  "Max": np.max(values),
        "P25":  np.percentile(values, 25),
        "P50":  np.percentile(values, 50),
        "P75":  np.percentile(values, 75),
        "Skew": skew(values),    "Kurt": kurtosis(values)
    }

stats_df = pd.DataFrame(stats).T.round(1)
print(tabulate(stats_df, headers="keys", stralign="right", floatfmt=".1f"))

In [ ]:
# 箱线图
fig, axes = plt.subplots(1, 4, figsize=(8, 2))
for ax, (label, values) in zip(axes, data.items()):
    sns.boxplot(y=values, ax=ax, color="gold")
    ax.set_title(label, fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

# 小提琴图
fig, axes = plt.subplots(1, 4, figsize=(8, 2))
for ax, (label, values) in zip(axes, data.items()):
    sns.violinplot(y=values, ax=ax, color="gold")
    ax.set_title(label, fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 3.6 应用实例：上证综合指数收益率年度分布

In [ ]:
import pandas as pd
import akshare as ak

sz_index = ak.stock_zh_index_daily(symbol="sh000001")
sz_index.rename(columns={'date': 'day', 'close': 'close'}, inplace=True)
sz_index['day'] = pd.to_datetime(sz_index['day'])
sz_index['daily_return'] = sz_index['close'].pct_change()
sz_index['year'] = sz_index['day'].dt.year
sz_index.drop(columns=['open', 'high', 'low'], inplace=True)

print(sz_index.head(3))
print('-' * 50)
print(sz_index.tail(3))

接下来，我们挑选几个特定的年份，绘制其日收益率的箱型图。

注意，此处我们使用的是 `seaborn` 库中的 `boxplot()` 函数，而不是 `matplotlib` 中的 `boxplot()` 函数。前者可以更好地处理数据的分组和分类，并且提供了更多的可视化选项。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

selected_years = [1997, 2005, 2006, 2007, 2014, 2015, 2021, 2024]
filtered_data = sz_index[sz_index['year'].isin(selected_years)]

plt.figure(figsize=(6, 3))
sns.boxplot(x='year', y='daily_return', data=filtered_data, palette='Set3')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

该图为多个特定年份的上证综合指数日收益率的箱线图，展示了每年交易日中日收益率的分布情况。

从**中位数线**（箱体中间的横线）来看：

- 中位数大于零：1997 年、2006 年、2007 年和 2015 年的中位收益率明显高于零，说明这些年份中有**超过一半的交易日呈现正收益**，整体市场偏强。
- 中位数接近于零：2014 年、2021 年和 2024 年中位数趋近于零，意味着正负收益天数接近持平。
- 中位数小于零：2005 年中位收益率低于零，表明该年中**大部分交易日处于负收益区间**，市场情绪低迷。

从箱体的**高度**（即 $\text{IQR} = Q_3 - Q_1$）和**胡须长度**来看：
- 1997 年、2007 年和 2015 年的箱体高度较高，且胡须较长，表明这几年的年内收益波动性较大，市场情绪起伏明显。
- 2021 年的箱体很低，胡须也很短，上下胡须外侧的离群值也很少，该年市场整体波动性较小。

从**离群点**来看：
- 1997 年的箱型图中，上下胡须外侧的离群值点都比较多，说明当年市场波动剧烈，存在较多极端收益的交易日（彼时尚无 10% 日内涨跌幅限制，市场波动性更大）。
- 2007 年和 2015 年的分布特征非常相似，都是在下胡须方向上有较多的离群点，说明存在较多单日大幅下跌的情况。

**特定年份分析：**

- **2005 年 vs 2006 年**：2005 年的箱体整体较低，中位数为负，反映当年大部分交易日处于负收益区间，市场情绪低迷；2006 年则大为反转，中位数跃升至零之上，显示出牛市初期的积极走势，这一变化与当年「股权分置改革推进、人民币升值预期增强」等政策背景密切相关。

- **2024 年**：箱体高度极窄，即 $Q_3$ 与 $Q_1$ 非常接近，说明日收益率的**四分位间距（IQR）很小**，波动性低；同时，存在较多离群点分布在上下两侧，提示虽然整体震荡区间狭窄，但**偶发性的大涨或大跌依然存在**，这可能与 AI、芯片等概念股轮动剧烈，但整体指数运行平稳有关。

下图呈现了几个特定年份的沪市综合指数的时序图，大家可以挑选一些年份，将其收盘价的时序图与上图中对应年份的箱型图进行对比，以便更深入地理解箱型图的含义。

![](figs/sz_index_daily_return_selected_years.png)

In [ ]:
# 小提琴图
plt.figure(figsize=(6, 3))
sns.violinplot(x='year', y='daily_return', data=filtered_data, palette='Set3')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3.7 箱线图与小提琴图的对比示例

In [ ]:
# https://matplotlib.org/stable/gallery/statistics/boxplot_vs_violin.html

import matplotlib.pyplot as plt
import numpy as np

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(9, 4))
np.random.seed(19680801)

all_data = [np.random.normal(0, std, 100) for std in range(6, 10)]

axs[0].violinplot(all_data, showmeans=False, showmedians=True)
axs[0].set_title('Violin plot')

axs[1].boxplot(all_data)
axs[1].set_title('Box plot')

for ax in axs:
    ax.yaxis.grid(True)
    ax.set_xticks([y + 1 for y in range(len(all_data))],
                  labels=['x1', 'x2', 'x3', 'x4'])
    ax.set_xlabel('Four separate samples')
    ax.set_ylabel('Observed values')

plt.show()

## 3.8 带散点的箱线图（Jitter Plot）

通过添加 stripplot，可以在展示箱线图汇总信息的同时，显示所有原始观测点。

> Source: [Hidden Data Under Boxplot](https://python-graph-gallery.com/39-hidden-data-under-boxplot/)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

a = pd.DataFrame({'group': np.repeat('A', 500), 'value': np.random.normal(10, 5, 500)})
b = pd.DataFrame({'group': np.repeat('B', 500), 'value': np.random.normal(13, 1.2, 500)})
c = pd.DataFrame({'group': np.repeat('B', 500), 'value': np.random.normal(18, 1.2, 500)})
d = pd.DataFrame({'group': np.repeat('C', 20),  'value': np.random.normal(25, 4, 20)})
e = pd.DataFrame({'group': np.repeat('D', 100), 'value': np.random.uniform(12, size=100)})
df = pd.concat([a, b, c, d, e])

ax = sns.boxplot(x='group', y='value', data=df)
ax = sns.stripplot(x='group', y='value', data=df, color="orange", jitter=0.2, size=2.5)
plt.title("Boxplot with jitter", loc="left")
plt.show()

## 3.9 小提琴图：两个组别

- [seaborn.violinplot](https://seaborn.pydata.org/generated/seaborn.violinplot.html)

In [ ]:
import seaborn as sns

df = sns.load_dataset("titanic")
sns.violinplot(x=df["age"])
sns.violinplot(data=df, x="class", y="age", hue="alive", fill=False)

In [ ]:
sns.violinplot(data=df, x="class", y="age", hue="alive", split=True, inner="quart")

## 3.10 参考资料

- [Matplotlib Documentation](https://matplotlib.org/stable/gallery/statistics/boxplot_demo.html)
- [Matplotlib - Boxplot vs Violin](https://matplotlib.org/stable/gallery/statistics/boxplot_vs_violin.html)
- [Seaborn - Visualizing distributions of data](https://seaborn.pydata.org/tutorial/distributions.html)
- [Seaborn - Visualizing categorical data](https://seaborn.pydata.org/tutorial/categorical.html)
- [Seaborn Boxplot](https://seaborn.pydata.org/tutorial/boxplot.html)
- [Seaborn Violinplot](https://seaborn.pydata.org/generated/seaborn.violinplot.html)
- [Python Graph Gallery: Hidden Data Under Boxplot](https://python-graph-gallery.com/39-hidden-data-under-boxplot/)